# DDPM Task 2 — Analysis

Reads precomputed results already synced to `gdrive:lab1-analysis` by `run_analysis.py` (run on a rented GPU pod, not here) plus a couple of lightweight items still read directly from `gdrive:lab1-ckpts` (training-time trajectory PNGs and `train_state.pt` loss histories — both small, no GPU needed). No sampling or FID computation happens in this notebook; it's plotting and table-building only.

1. Table — final FID per config (the graded "best FID" number)
2. Figure — scheduler trajectory comparison (linear/quad/cosine, noise predictor)
3. Figure — predictor sample comparison (noise/x0/mean, linear scheduler, 8 samples each)
4. Figure + Table — FID vs. training step, all configs, plus convergence-speed table
5. Figure — sample quality progression over training (linear+noise)
6. Figure — training loss curves overlaid, all configs
7. Figure — worst samples per config (least realistic, by nearest-neighbor distance to real Inception features)

**Sample-count note:** the FID curve's non-final points use 100 generated samples each (enough to show the trend cheaply across ~40 checkpoints); each config's *final* checkpoint uses 500 samples (the project's own FID convention, matching the README default) — that's the number in Table 1 and the number that should be screenshotted for submission. The two aren't directly comparable point-for-point; Figure 3 marks the final point distinctly for this reason.

Run cells top to bottom. Requires `lab1-ckpts` and `lab1-analysis` to already be in your Google Drive.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, glob, re
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image

DRIVE_CKPT_ROOT = "/content/drive/MyDrive/lab1-ckpts"      # training-time trajectory PNGs, train_state.pt
ANALYSIS_ROOT = "/content/drive/MyDrive/lab1-analysis"     # everything run_analysis.py precomputed

# label -> (beta scheduler mode, predictor parameterization)
CONFIGS = {
    "linear_noise": {"mode": "linear", "predictor": "noise"},
    "quad_noise":   {"mode": "quad",   "predictor": "noise"},
    "cosine_noise": {"mode": "cosine", "predictor": "noise"},
    "linear_x0":    {"mode": "linear", "predictor": "x0"},
    "linear_mean":  {"mode": "linear", "predictor": "mean"},
}
PREDICTOR_LABELS = ["linear_noise", "linear_x0", "linear_mean"]

def ckpt_dir(label):
    cfg = CONFIGS[label]
    return f"{DRIVE_CKPT_ROOT}/{label}/predictor_{cfg['predictor']}/beta_{cfg['mode']}"

def latest_traj(label):
    d = ckpt_dir(label)
    files = glob.glob(f"{d}/step=*-traj.png")
    steps = [int(re.search(r"step=(\d+)-traj", f).group(1)) for f in files]
    best = max(steps)
    return f"{d}/step={best}-traj.png", best

def load_png_grid(paths, ax_row):
    """Displays a sorted list of image paths across a row of axes."""
    for ax, path in zip(ax_row, paths):
        ax.imshow(Image.open(path))
        ax.axis("off")
    for ax in ax_row[len(paths):]:
        ax.axis("off")

with open(f"{ANALYSIS_ROOT}/fid_curve_results.json") as f:
    fid_curves_raw = json.load(f)
# {label: {step (int): fid}}, sorted ascending by step
fid_curves = {label: dict(sorted(((int(s), fid) for s, fid in steps.items())))
              for label, steps in fid_curves_raw.items()}

print("Configs:", list(CONFIGS))
print("FID results loaded for:", {l: len(c) for l, c in fid_curves.items()})

## 1. Table — final FID per config

The graded "best FID" number: the final point on each config's FID curve, computed by `run_analysis.py` with 500 samples (the README's own convention) against each config's actual final checkpoint. Also doubles as the scheduler/predictor comparison table.

In [ ]:
rows = []
for label, cfg in CONFIGS.items():
    curve = fid_curves[label]
    final_step = max(curve)
    fid = curve[final_step]
    rows.append({"label": label, "scheduler": cfg["mode"], "predictor": cfg["predictor"],
                 "final_step": final_step, "FID": fid})
    print(f"{label}: step={final_step} FID={fid:.4f}")

df_final = pd.DataFrame(rows)
df_final

## 2. Figure — scheduler trajectory comparison

`linear` vs `quad` vs `cosine` (all noise predictor). Reuses the trajectory strips already rendered during training — note these are from the *last periodic checkpoint before* the final step (e.g. step 98000 not 100000), since trajectory images are only written inside the periodic log_interval block, which never fires exactly at the final step.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 12))
for ax, label in zip(axs, ["linear_noise", "quad_noise", "cosine_noise"]):
    path, step = latest_traj(label)
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title(f"{CONFIGS[label]['mode']} scheduler (trajectory at step {step})", fontsize=14)
plt.tight_layout()
plt.savefig("fig1_scheduler_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Figure — predictor sample comparison

`noise` vs `x0` vs `mean` (all linear scheduler), 8 samples each — precomputed by `run_analysis.py` from each config's final checkpoint and synced to `gdrive:lab1-analysis/predictor_samples/{predictor}/`.

In [ ]:
predictor_labels = ["linear_noise", "linear_x0", "linear_mean"]
fig, axs = plt.subplots(3, 8, figsize=(24, 9.5))

for row, label in enumerate(predictor_labels):
    predictor = CONFIGS[label]["predictor"]
    paths = sorted(glob.glob(f"{ANALYSIS_ROOT}/predictor_samples/{predictor}/*.png"),
                    key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
    load_png_grid(paths, axs[row])
    axs[row, 0].text(-0.3, 0.5, predictor, fontsize=16, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("Predictor comparison: 8 samples each (linear scheduler)", fontsize=16)
plt.tight_layout()
plt.savefig("fig2_predictor_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Figure + Table — FID vs. training step (all checkpoints, all configs)

The result only possible because every checkpoint was kept, not just the final one: shows *when* each config actually converges, and whether the final checkpoint is even the best one. Precomputed by `run_analysis.py` at ~8 evenly-spaced checkpoints per config (100 samples each — enough to show the trend, not publication-grade) plus each config's true final checkpoint (500 samples, the headline number). **The final point uses a different, larger sample count than the rest of the curve** — it's marked with a star and excluded from the "first step below threshold" table below, so it isn't mistaken for just another curve point.

In [ ]:
plt.figure(figsize=(11, 6))
for label, curve in fid_curves.items():
    steps_ = list(curve.keys())
    fids_ = list(curve.values())
    final_step = max(steps_)
    line, = plt.plot(steps_, fids_, marker="o", markersize=5, label=label)
    # final point used 500 samples (the headline number), the rest used 100 -- mark it
    # distinctly so it isn't read as just another point on the same trend
    plt.plot(final_step, curve[final_step], marker="*", markersize=16,
              color=line.get_color(), markeredgecolor="black", linestyle="none")
plt.xlabel("Training step")
plt.ylabel("FID (lower is better)")
plt.yscale("log")
plt.title("FID vs. training step, all 5 configs\n(\u2605 = final checkpoint, 500 samples; \u25cf = curve point, 100 samples)")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.savefig("fig3_fid_vs_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Convergence-speed table: earliest step each config's FID first dropped below a threshold.
# Uses only the 100-sample curve points (excludes the 500-sample final point) so the
# threshold comparison is apples-to-apples across every row.
conv_rows = []
for label, curve in fid_curves.items():
    final_step = max(curve)
    curve_only = {s: f for s, f in curve.items() if s != final_step}
    first_below_20 = next((s for s, f in sorted(curve_only.items()) if f < 20), None)
    first_below_15 = next((s for s, f in sorted(curve_only.items()) if f < 15), None)
    best_step, best_fid = min(curve_only.items(), key=lambda sf: sf[1])
    conv_rows.append({
        "label": label,
        "first_step_FID<20": first_below_20,
        "first_step_FID<15": first_below_15,
        "best_curve_step": best_step,
        "best_curve_FID": round(best_fid, 4),
        "final_step": final_step,
        "final_FID": round(curve[final_step], 4),
    })
df_convergence = pd.DataFrame(conv_rows)
df_convergence

## 5. Figure — sample quality progression (linear + noise)

The longest, most-checkpointed run — visual complement to the FID curve above, showing the same convergence directly in image space. Precomputed by `run_analysis.py` at ~5 evenly-spaced steps, synced to `gdrive:lab1-analysis/progression/step={step}/`.

In [ ]:
prog_dirs = sorted(glob.glob(f"{ANALYSIS_ROOT}/progression/step=*"),
                    key=lambda d: int(re.search(r"step=(\d+)", d).group(1)))
chosen_steps = [int(re.search(r"step=(\d+)", d).group(1)) for d in prog_dirs]

fig, axs = plt.subplots(len(prog_dirs), 8, figsize=(24, 3 * len(prog_dirs)))
if len(prog_dirs) == 1:
    axs = axs[None, :]
for row, (d, step) in enumerate(zip(prog_dirs, chosen_steps)):
    paths = sorted(glob.glob(f"{d}/*.png"), key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
    load_png_grid(paths, axs[row])
    axs[row, 0].text(-0.3, 0.5, f"step {step}", fontsize=14, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("linear + noise: sample quality over training", fontsize=16)
plt.tight_layout()
plt.savefig("fig4_progression.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Figure — training loss curves overlaid

Pulled directly from each config's `train_state.pt` (`losses` list, saved throughout training), smoothed with a moving average since raw per-step MSE loss is noisy.

In [ ]:
plt.figure(figsize=(11, 6))
WINDOW = 200
for label in CONFIGS:
    d = ckpt_dir(label)
    ts = torch.load(f"{d}/train_state.pt", map_location="cpu", weights_only=False)
    losses = np.array(ts["losses"])
    if len(losses) >= WINDOW:
        smoothed = np.convolve(losses, np.ones(WINDOW) / WINDOW, mode="valid")
    else:
        smoothed = losses
    plt.plot(smoothed, label=label, alpha=0.85)
plt.xlabel("Training step")
plt.ylabel(f"Loss ({WINDOW}-step moving average)")
plt.title("Training loss curves, all 5 configs")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Figure — worst samples per config

FID is a distributional statistic with no per-image value, so "worst sample" needs a proxy: each config's final-checkpoint generations were ranked by nearest-neighbor distance to the real Inception activation cloud (a standard stand-in for "how unrealistic is this one image"). Precomputed by `run_analysis.py`, synced to `gdrive:lab1-analysis/worst_samples/{label}/` with the underlying scores in `error_analysis_{label}.json`. Shows the 6 worst per config.

In [ ]:
N_WORST_SHOWN = 6
fig, axs = plt.subplots(len(CONFIGS), N_WORST_SHOWN, figsize=(3 * N_WORST_SHOWN, 3 * len(CONFIGS)))

for row, label in enumerate(CONFIGS):
    with open(f"{ANALYSIS_ROOT}/error_analysis_{label}.json") as f:
        err = json.load(f)
    worst = err["worst"][:N_WORST_SHOWN]  # already sorted worst-first by run_analysis.py
    paths = [f"{ANALYSIS_ROOT}/worst_samples/{label}/{w['file']}" for w in worst]
    load_png_grid(paths, axs[row])
    for col, w in enumerate(worst):
        axs[row, col].set_title(f"d={w['nn_distance']:.1f}", fontsize=10)
    axs[row, 0].text(-0.3, 0.5, label, fontsize=13, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("Worst samples per config (by nearest-neighbor distance to real Inception features)", fontsize=16)
plt.tight_layout()
plt.savefig("fig6_worst_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save everything for the report

Copies the final table CSVs and all figure PNGs into Drive so they survive the Colab runtime shutting down.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/lab1-analysis"
os.makedirs(OUT_DIR, exist_ok=True)

df_final.to_csv(f"{OUT_DIR}/table1_final_fid.csv", index=False)
df_convergence.to_csv(f"{OUT_DIR}/table2_convergence.csv", index=False)

import shutil
for f in glob.glob("fig*.png"):
    shutil.copy(f, OUT_DIR)

print("Saved to", OUT_DIR)
print(os.listdir(OUT_DIR))